# Proyecto: Red Neuronal Convolucional (CNN)

**Autora:** Ashly Umanzor - Ingeniera en Sistemas 

En las siguientes celdas se presenta el desarrollo paso a paso de una red neuronal convolucional, desde la preparación de los datos hasta el entrenamiento, evaluación y análisis del modelo.


## Descripción del Proyecto
Este proyecto consiste en el desarrollo de una Red Neuronal Convolucional (CNN) para la clasificación de imágenes de animales. El modelo fue entrenado con cinco clases previamente definidas, permitiendo identificar si una imagen pertenece a alguna de ellas.

El objetivo principal es construir un modelo capaz de reconocer patrones visuales distintivos de cada categoría y clasificar correctamente las imágenes. Además, se contempla la detección de imágenes que no correspondan a ninguna de las clases entrenadas, mediante una validación basada en el nivel de confianza del modelo.


## Celda N°1 – Imports, configuración, rutas y detección de clases

In [ ]:
import os
from pathlib import Path
import random

import numpy as np
from PIL import Image

# Uso de semilla aleatoria: Esto asegura que se reproduzca el proceso de la mejor manera.
SEMILLA_ALEATORIA = 42 
random.seed(SEMILLA_ALEATORIA)
np.random.seed(SEMILLA_ALEATORIA)


# Rutas y Datos
RUTA_DATOS = Path("/content/drive/MyDrive/data/animales")

nombresClases = sorted(
    [carpeta.name for carpeta in RUTA_DATOS.iterdir() if carpeta.is_dir()]
)
numeroClases = len(nombresClases)

print("Clases detectadas:", nombresClases)
print("Número de clases:", numeroClases)


# Parámetros de las imágenes
ALTURA_IMAGEN = 32
ANCHURA_IMAGEN = 32
NUM_CANALES = 1


# Hiperparámetros de la red
NEURONAS_CAPA_OCULTA = 64
NUM_FILTROS_CONV = 8           # Este correponde al número de filtros en la capa convolucional (Caracteristicas que puede aprender)
TAMANO_KERNEL = (3, 3)
TAMANO_POOL = (2, 2)
STRIDE_POOL = 2                # Número de Pixeles a leer.

TASA_APRENDIZAJE = 0.005
EPOCAS = 50
TAMANO_LOTE = 16 #Mide la eficiencia del entrenamiento

# Umbral para "ninguna de las anteriores"
UMBRAL_NINGUNA = 0.6


Clases detectadas: ['Aves', 'Caballos', 'Gatos', 'Peces', 'Perros']
Número de clases: 5


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Celda N°2 – Carga y preprocesamiento de imágenes

In [ ]:
def cargar_imagenes_desde_carpetas(rutaRaiz, listaNombresClases, alturaImagen, anchuraImagen):
    """
    Recorre rutaRaiz, donde cada subcarpeta representa una clase.

    Devuelve:
    - matrizImagenes: array de shape (N, alturaImagen, anchuraImagen, 1)
      con imágenes en escala de grises normalizadas en [0, 1].
    - etiquetas: array de enteros de shape (N,) con la clase de cada imagen.
    """
    listaImagenes = []
    listaEtiquetas = []

    for indiceClase, nombreClase in enumerate(listaNombresClases):
        rutaClase = rutaRaiz / nombreClase
        archivosClase = os.listdir(rutaClase)
        print(f"Clase '{nombreClase}' - número de archivos:", len(archivosClase))

        for nombreArchivo in archivosClase:
            rutaArchivo = rutaClase / nombreArchivo

            # Intentamos abrir el archivo como imagen
            try:
                imagen = Image.open(rutaArchivo).convert("L")  # "L" = escala de grises
            except Exception:
                # Si no se puede abrir como imagen, lo ignoramos
                continue

            # Redimensionar la imagen al tamaño estándar
            imagen = imagen.resize((anchuraImagen, alturaImagen))

            # Convertir a array NumPy (float32)
            arregloImagen = np.array(imagen, dtype=np.float32)

            # Normalizar pixeles a rango [0, 1]
            arregloImagen /= 255.0

            # Añadir dimensión de canal: (altura, anchura) donde (altura, anchura, 1)
            arregloImagen = np.expand_dims(arregloImagen, axis=-1)

            listaImagenes.append(arregloImagen)
            listaEtiquetas.append(indiceClase)

    matrizImagenes = np.array(listaImagenes, dtype=np.float32)
    etiquetas = np.array(listaEtiquetas, dtype=np.int64)

    print("Total de imágenes cargadas:", matrizImagenes.shape[0])
    return matrizImagenes, etiquetas

matrizImagenes, etiquetas = cargar_imagenes_desde_carpetas(
    rutaRaiz=RUTA_DATOS,
    listaNombresClases=nombresClases,
    alturaImagen=ALTURA_IMAGEN,
    anchuraImagen=ANCHURA_IMAGEN
)

print("Shape de matrizImagenes:", matrizImagenes.shape)  # (N, 32, 32, 1)
print("Shape de etiquetas:", etiquetas.shape)            # (N,)


Clase 'Aves' - número de archivos: 210
Clase 'Caballos' - número de archivos: 210
Clase 'Gatos' - número de archivos: 210
Clase 'Peces' - número de archivos: 210
Clase 'Perros' - número de archivos: 210
Total de imágenes cargadas: 1050
Shape de matrizImagenes: (1050, 32, 32, 1)
Shape de etiquetas: (1050,)


## Celda N°3 – División entrenamiento/validación


Variables clave importantes a considerar aquí:
* **imagenesEntrenamiento, etiquetasEntrenamiento:** corresponde a los datos que usaremos para ajustar los pesos de la red.
* **imagenesValidacion, etiquetasValidacion:** corresponde a los datos que usaremos para medir qué tan bien generaliza.

In [ ]:
def dividir_entrenamiento_validacion(matrizImagenes, etiquetas, proporcionValidacion=0.2):
    """
    Divide el dataset en entrenamiento y validación.
    Mezcla los datos antes de dividir.
    """
    numeroMuestras = matrizImagenes.shape[0]
    indices = np.arange(numeroMuestras)
    np.random.shuffle(indices)

    indiceCorte = int(numeroMuestras * (1 - proporcionValidacion))

    indicesEntrenamiento = indices[:indiceCorte]
    indicesValidacion = indices[indiceCorte:]

    imagenesEntrenamiento = matrizImagenes[indicesEntrenamiento]
    etiquetasEntrenamiento = etiquetas[indicesEntrenamiento]

    imagenesValidacion = matrizImagenes[indicesValidacion]
    etiquetasValidacion = etiquetas[indicesValidacion]

    return imagenesEntrenamiento, etiquetasEntrenamiento, imagenesValidacion, etiquetasValidacion

def codificar_one_hot(etiquetasEnteras, numeroClases): #Convierto los datos categoricos en un formato númerico.
    """
    Convierte un vector de etiquetas (N,) en una matriz one-hot (N, numeroClases).

    Ejemplo:
    Si numeroClases = 3 y etiquetas = [1, 0] -> [[0,1,0],[1,0,0]]
    """
    numeroMuestras = etiquetasEnteras.shape[0]
    matrizOneHot = np.zeros((numeroMuestras, numeroClases), dtype=np.float32)
    matrizOneHot[np.arange(numeroMuestras), etiquetasEnteras] = 1.0
    return matrizOneHot

imagenesEntrenamiento, etiquetasEntrenamiento, imagenesValidacion, etiquetasValidacion = \
    dividir_entrenamiento_validacion(
        matrizImagenes=matrizImagenes,
        etiquetas=etiquetas,
        proporcionValidacion=0.2
    )

print("Tamaño entrenamiento:", imagenesEntrenamiento.shape[0])
print("Tamaño validación:", imagenesValidacion.shape[0])


Tamaño entrenamiento: 840
Tamaño validación: 210


## Celda N°4 – Funciones de activación (ReLU)

Observación Importante:
* ReLU introduce no linealidad, lo que permite a la red aprender funciones más complejas que una simple combinación lineal.

In [ ]:
def funcion_relu(xTensor):
    """
    Función de activación ReLU:
    - Si x > 0 -> x
    - Si x <= 0 -> 0

    Se aplica elemento a elemento sobre el tensor de entrada.
    """
    return np.maximum(0, xTensor)

def derivada_relu(xTensor):
    """
    Derivada de ReLU:
    - 1 cuando x > 0
    - 0 cuando x <= 0

    Se usa en la propagación hacia atrás (backpropagation).
    """
    mascara = (xTensor > 0).astype(np.float32)
    return mascara


## Celda N°5 – Clases de capas (Convolución, MaxPooling, Capa de aplanado, Capa Densa Oculta - Salida,FullyConnected, ReLU)

A partir de aquí se construye la CNN completa:

In [ ]:
class CapaConvolucional:
    """
    Capa convolucional 2D muy simple:
    - Imágenes en escala de grises: shape (N, H, W, 1)
    - Filtros: shape (numFiltros, kH, kW)
    - Convolución 'válida' (sin padding), stride = 1
    """

    def __init__(self, numFiltros, tamanoKernel):
        self.numFiltros = numFiltros
        self.tamanoKernel = tamanoKernel  # (kH, kW)

        kH, kW = tamanoKernel

        # Inicializamos filtros y sesgos (pequeños valores aleatorios), aqui es donde se encuentran los pesos de la red.
        escala = np.sqrt(2.0 / (kH * kW))
        self.filtros = np.random.randn(numFiltros, kH, kW) * escala
        self.sesgos = np.zeros((numFiltros,), dtype=np.float32)

        # Para guardar durante forward
        self.inputTensor = None

        # Gradientes que definí:
        self.gradFiltros = None
        self.gradSesgos = None

    def forward_pass(self, inputTensor):
        """
        inputTensor: (N, H, W, 1)
        Devuelve:
        outputTensor: (N, H_out, W_out, numFiltros)
        """
        self.inputTensor = inputTensor
        batchSize, altura, anchura, canales = inputTensor.shape
        kH, kW = self.tamanoKernel

        alturaSalida = altura - kH + 1
        anchuraSalida = anchura - kW + 1

        outputTensor = np.zeros(
            (batchSize, alturaSalida, anchuraSalida, self.numFiltros),
            dtype=np.float32
        )

        # Convolución por cada filtro (implementación con bucles para entender la lógica)
        for n in range(batchSize):
            for f in range(self.numFiltros):
                filtro = self.filtros[f]
                sesgo = self.sesgos[f]
                for i in range(alturaSalida):
                    for j in range(anchuraSalida):
                        region = inputTensor[n, i:i+kH, j:j+kW, 0]  # región de la imagen
                        outputTensor[n, i, j, f] = np.sum(region * filtro) + sesgo

        return outputTensor

    def backward_pass(self, gradOutput):
        """
        gradOutput: gradiente de la pérdida con respecto a la salida de esta capa
        shape: (N, H_out, W_out, numFiltros)

        Devuelve:
        gradInput: gradiente con respecto a la entrada (N, H, W, 1)
        """
        inputTensor = self.inputTensor
        batchSize, altura, anchura, canales = inputTensor.shape
        kH, kW = self.tamanoKernel
        _, alturaSalida, anchuraSalida, _ = gradOutput.shape

        gradInput = np.zeros_like(inputTensor, dtype=np.float32)
        self.gradFiltros = np.zeros_like(self.filtros, dtype=np.float32)
        self.gradSesgos = np.zeros_like(self.sesgos, dtype=np.float32)

        # Recorremos igual que en forward, pero ahora calculando gradientes
        for n in range(batchSize):
            for f in range(self.numFiltros):
                for i in range(alturaSalida):
                    for j in range(anchuraSalida):
                        gradValor = gradOutput[n, i, j, f]
                        region = inputTensor[n, i:i+kH, j:j+kW, 0]

                        # Gradiente de los filtros
                        self.gradFiltros[f] += gradValor * region

                        # Gradiente de la entrada
                        gradInput[n, i:i+kH, j:j+kW, 0] += gradValor * self.filtros[f]

                        # Gradiente de los sesgos
                        self.gradSesgos[f] += gradValor

        return gradInput

    def update_params(self, learningRate):
        self.filtros -= learningRate * self.gradFiltros
        self.sesgos -= learningRate * self.gradSesgos


class CapaMaxPooling:
    """
    Capa de Max Pooling 2D:
    - Ventana poolSize (pH, pW)
    - Stride = stride
    Trabaja sobre tensores (N, H, W, C).
    """

    def __init__(self, poolSize, stride):
        self.poolSize = poolSize  # (pH, pW)
        self.stride = stride

        self.inputTensor = None
        self.mascaraMaximos = None

    def forward_pass(self, inputTensor):
        """
        inputTensor: (N, H, W, C)
        Devuelve:
        outputTensor: (N, H_out, W_out, C)
        """
        self.inputTensor = inputTensor
        batchSize, altura, anchura, canales = inputTensor.shape
        pH, pW = self.poolSize

        alturaSalida = (altura - pH) // self.stride + 1
        anchuraSalida = (anchura - pW) // self.stride + 1

        outputTensor = np.zeros(
            (batchSize, alturaSalida, anchuraSalida, canales),
            dtype=np.float32
        )
        self.mascaraMaximos = np.zeros_like(inputTensor, dtype=np.float32)

        # Recorremos cada ventana y tomamos el máximo
        for n in range(batchSize):
            for c in range(canales):
                for i in range(alturaSalida):
                    for j in range(anchuraSalida):
                        hInicio = i * self.stride
                        wInicio = j * self.stride
                        region = inputTensor[n, hInicio:hInicio+pH, wInicio:wInicio+pW, c]
                        maximo = np.max(region)
                        outputTensor[n, i, j, c] = maximo

                        # Guardamos dónde estaba el máximo para el backward
                        mascaraRegion = (region == maximo).astype(np.float32)
                        self.mascaraMaximos[n, hInicio:hInicio+pH, wInicio:wInicio+pW, c] = mascaraRegion

        return outputTensor

    def backward_pass(self, gradOutput):
        """
        gradOutput: (N, H_out, W_out, C)
        Devuelve:
        gradInput: gradiente hacia la entrada (N, H, W, C)
        """
        batchSize, altura, anchura, canales = self.inputTensor.shape
        pH, pW = self.poolSize
        _, alturaSalida, anchuraSalida, _ = gradOutput.shape

        gradInput = np.zeros_like(self.inputTensor, dtype=np.float32)

        for n in range(batchSize):
            for c in range(canales):
                for i in range(alturaSalida):
                    for j in range(anchuraSalida):
                        hInicio = i * self.stride
                        wInicio = j * self.stride

                        # El gradiente se reparte solo donde estaba el máximo
                        gradRegion = self.mascaraMaximos[n, hInicio:hInicio+pH, wInicio:wInicio+pW, c] \
                                     * gradOutput[n, i, j, c]
                        gradInput[n, hInicio:hInicio+pH, wInicio:wInicio+pW, c] += gradRegion

        return gradInput

    def update_params(self, learningRate):
        # No tiene parámetros entrenables
        pass


class CapaAplanado:
    """
    Capa que aplasta (flatten) un tensor (N, H, W, C) a (N, H*W*C).
    """

    def __init__(self):
        self.inputShape = None

    def forward_pass(self, inputTensor):
        self.inputShape = inputTensor.shape  # (N, H, W, C)
        batchSize = self.inputShape[0]
        outputTensor = inputTensor.reshape(batchSize, -1)
        return outputTensor

    def backward_pass(self, gradOutput):
        """
        gradOutput: (N, H*W*C)
        """
        gradInput = gradOutput.reshape(self.inputShape)
        return gradInput

    def update_params(self, learningRate):
        pass


class CapaDensa:
    """
    Capa totalmente conectada (FC):
    - Entrada: (N, inputSize)
    - Salida: (N, outputSize)
    """

    def __init__(self, inputSize, outputSize):
        self.inputSize = inputSize
        self.outputSize = outputSize

        # Inicialización sencilla (tipo He) para ReLU
        escala = np.sqrt(2.0 / inputSize)
        self.pesos = np.random.randn(inputSize, outputSize) * escala
        self.sesgos = np.zeros((1, outputSize), dtype=np.float32)

        self.inputTensor = None
        self.gradPesos = None  #AQUI SE LE COLOCO LOS PESOS
        self.gradSesgos = None

    def forward_pass(self, inputTensor):
        """
        inputTensor: (N, inputSize)
        """
        self.inputTensor = inputTensor
        outputTensor = inputTensor @ self.pesos + self.sesgos
        return outputTensor

    def backward_pass(self, gradOutput):
        """
        gradOutput: gradiente de la pérdida respecto a la salida de la capa (N, outputSize)
        Devuelve:
        gradInput: (N, inputSize)
        """
        self.gradPesos = self.inputTensor.T @ gradOutput
        self.gradSesgos = np.sum(gradOutput, axis=0, keepdims=True)
        gradInput = gradOutput @ self.pesos.T
        return gradInput

    def update_params(self, learningRate):
        self.pesos -= learningRate * self.gradPesos
        self.sesgos -= learningRate * self.gradSesgos


class CapaReLU:
    """
    Capa de activación ReLU como objeto separado.
    """

    def __init__(self):
        self.inputTensor = None

    def forward_pass(self, inputTensor):
        self.inputTensor = inputTensor
        return funcion_relu(inputTensor)

    def backward_pass(self, gradOutput):
        gradInput = gradOutput * derivada_relu(self.inputTensor)
        return gradInput

    def update_params(self, learningRate):
        pass


## Celda N°6 – Función de pérdida: Softmax + Entropía Cruzada

In [ ]:
class FuncionSoftmaxEntropiaCruzada:
    """
    Combina Softmax y entropía cruzada.
    """

    def __init__(self):
        self.probabilidades = None
        self.etiquetasOneHot = None

    def forward(self, logitsTensor, etiquetasOneHot):
        """
        logitsTensor: (N, numeroClases) -> salidas lineales de la última capa densa
        etiquetasOneHot: (N, numeroClases)
        Devuelve:
        perdidaMedia (escalar)
        """
        # Softmax estable numéricamente
        logitsAjustados = logitsTensor - np.max(logitsTensor, axis=1, keepdims=True)
        expLogits = np.exp(logitsAjustados)
        probabilidades = expLogits / np.sum(expLogits, axis=1, keepdims=True)

        self.probabilidades = probabilidades
        self.etiquetasOneHot = etiquetasOneHot

        numeroMuestras = logitsTensor.shape[0]
        epsilon = 1e-15
        probabilidadesClipeadas = np.clip(probabilidades, epsilon, 1 - epsilon)
        logProbabilidades = -np.log(probabilidadesClipeadas)

        perdidaPorMuestra = np.sum(etiquetasOneHot * logProbabilidades, axis=1)
        perdidaMedia = np.mean(perdidaPorMuestra)

        return perdidaMedia

    def backward(self):
        """
        Devuelve el gradiente de la pérdida respecto a los logits:
        dL/dLogits = (probabilidades - etiquetasOneHot) / N
        """
        numeroMuestras = self.etiquetasOneHot.shape[0]
        gradLogits = (self.probabilidades - self.etiquetasOneHot) / numeroMuestras #Aqui hace el calculo del gradiente
        return gradLogits


## Celda N°7 – Clase ModeloCNN (modelo completo)

Observación Importante:
* Aquí se ensamblan todas las capas en la arquitectura mínima requerida:
  * Conv → ReLU → MaxPool → Flatten → Densa oculta (ReLU) → Densa salida.

In [ ]:
class ModeloCNN:
    """
    CNN simple (arquitectura mínima):
    Entrada (H, W, 1)
    -> CapaConvolucional (3x3, NUM_FILTROS_CONV filtros)
    -> CapaReLU
    -> CapaMaxPooling (2x2, stride 2)
    -> CapaAplanado
    -> CapaDensa oculta (NEURONAS_CAPA_OCULTA)
    -> CapaReLU
    -> CapaDensa salida (numeroClases)
    """

    def __init__(self, inputShape, numeroClases):
        self.inputShape = inputShape  # (H, W, C)
        self.numeroClases = numeroClases

        # Definimos capas convolucionales / pooling / flatten
        self.convLayer = CapaConvolucional(numFiltros=NUM_FILTROS_CONV,
                                           tamanoKernel=TAMANO_KERNEL)
        self.relu1 = CapaReLU()
        self.poolLayer = CapaMaxPooling(poolSize=TAMANO_POOL,
                                        stride=STRIDE_POOL)
        self.flattenLayer = CapaAplanado()

        # Para determinar el tamaño de entrada de la capa densa,
        # pasamos un tensor dummy por estas capas
        dummyInput = np.zeros((1,) + inputShape, dtype=np.float32)
        xTensor = self.convLayer.forward_pass(dummyInput)
        xTensor = self.relu1.forward_pass(xTensor)
        xTensor = self.poolLayer.forward_pass(xTensor)
        xTensor = self.flattenLayer.forward_pass(xTensor)
        tamanoFlatten = xTensor.shape[1]

        # Definimos capas fully connected
        self.fcOculta = CapaDensa(tamanoFlatten, NEURONAS_CAPA_OCULTA) #Aqui es donde se usan las neruonas de la capa oculta.
        self.relu2 = CapaReLU()
        self.fcSalida = CapaDensa(NEURONAS_CAPA_OCULTA, numeroClases)

        # Lista para recorrer todas las capas en orden
        self.listaCapas = [
            self.convLayer,
            self.relu1,
            self.poolLayer,
            self.flattenLayer,
            self.fcOculta,
            self.relu2,
            self.fcSalida
        ]

    def forward_pass(self, inputTensor):
        """
        Pasa el inputTensor por todas las capas en orden.
        Devuelve logits (sin softmax).
        """
        xTensor = inputTensor
        for capa in self.listaCapas:
            xTensor = capa.forward_pass(xTensor)
        return xTensor  # logits

    def backward_pass(self, gradOutput):
        """
        Propaga el gradiente hacia atrás por todas las capas (en orden inverso).
        """
        gradTensor = gradOutput
        for capa in reversed(self.listaCapas):
            gradTensor = capa.backward_pass(gradTensor)
        return gradTensor

    def update_params(self, learningRate):
        """
        Actualiza parámetros entrenables de cada capa.
        """
        for capa in self.listaCapas:
            capa.update_params(learningRate)

    def predecir_clase_probas(self, inputTensor):
        """
        Devuelve:
        - clasesPredichas: vector de enteros (N,)
        - probabilidades: matriz (N, numeroClases)
        """
        logitsTensor = self.forward_pass(inputTensor)

        # Softmax para convertir logits en probabilidades
        logitsAjustados = logitsTensor - np.max(logitsTensor, axis=1, keepdims=True)
        expLogits = np.exp(logitsAjustados)
        probabilidades = expLogits / np.sum(expLogits, axis=1, keepdims=True)

        clasesPredichas = np.argmax(probabilidades, axis=1)
        return clasesPredichas, probabilidades


## Celda N°8 – Entrenamiento de la CNN (forward, backward, actualización)


In [ ]:

# Celda 8: Entrenamiento de la CNN


def calcular_exactitud(modelo, imagenes, etiquetas):
    """
    Calcula la exactitud (accuracy) del modelo:
    porcentaje de etiquetas predichas correctamente.
    """
    clasesPredichas, _ = modelo.predecir_clase_probas(imagenes)
    exactitud = np.mean(clasesPredichas == etiquetas)
    return exactitud

# Creamos el modelo y la función de pérdida
inputShape = (ALTURA_IMAGEN, ANCHURA_IMAGEN, NUM_CANALES)
modeloCnn = ModeloCNN(inputShape=inputShape, numeroClases=numeroClases)
funcionPerdida = FuncionSoftmaxEntropiaCruzada()

numeroMuestrasTrain = imagenesEntrenamiento.shape[0]
numeroLotes = int(np.ceil(numeroMuestrasTrain / TAMANO_LOTE))

# Para guardar historial
historialPerdida = []
historialExactitudTrain = []
historialExactitudVal = []

for epoca in range(1, EPOCAS + 1):
    # Barajamos el conjunto de entrenamiento
    indices = np.arange(numeroMuestrasTrain)
    np.random.shuffle(indices)

    imagenesBarajadas = imagenesEntrenamiento[indices]
    etiquetasBarajadas = etiquetasEntrenamiento[indices]

    perdidaAcumulada = 0.0

    for indiceLote in range(numeroLotes):
        inicio = indiceLote * TAMANO_LOTE
        fin = min(inicio + TAMANO_LOTE, numeroMuestrasTrain)

        loteImagenes = imagenesBarajadas[inicio:fin]
        loteEtiquetas = etiquetasBarajadas[inicio:fin]

        # Forward pass por la CNN
        logitsLote = modeloCnn.forward_pass(loteImagenes)

        # One-hot de etiquetas
        loteEtiquetasOneHot = codificar_one_hot(loteEtiquetas, numeroClases)

        # Cálculo de la pérdida
        perdidaLote = funcionPerdida.forward(logitsLote, loteEtiquetasOneHot)
        perdidaAcumulada += perdidaLote

        # Backward: gradiente w.r.t. logits Aqui empieza el entrenamiento segun el BPro OJOOOOOOOOO
        gradLogits = funcionPerdida.backward()

        # Propagación hacia atrás en la CNN
        _ = modeloCnn.backward_pass(gradLogits)

        # Actualización de parámetros
        modeloCnn.update_params(TASA_APRENDIZAJE)

    perdidaMediaEpoca = perdidaAcumulada / numeroLotes

    # Exactitud en entrenamiento y validación
    exactitudTrain = calcular_exactitud(modeloCnn, imagenesEntrenamiento, etiquetasEntrenamiento)
    exactitudVal = calcular_exactitud(modeloCnn, imagenesValidacion, etiquetasValidacion)

    historialPerdida.append(perdidaMediaEpoca)
    historialExactitudTrain.append(exactitudTrain)
    historialExactitudVal.append(exactitudVal)

    print(
        f"Época {epoca:02d}/{EPOCAS} - "
        f"Pérdida: {perdidaMediaEpoca:.4f} - "
        f"Exactitud train: {exactitudTrain:.4f} - "
        f"Exactitud val: {exactitudVal:.4f}"
    )


Época 01/50 - Pérdida: 1.6911 - Exactitud train: 0.2690 - Exactitud val: 0.2619
Época 02/50 - Pérdida: 1.5876 - Exactitud train: 0.3155 - Exactitud val: 0.3143
Época 03/50 - Pérdida: 1.5447 - Exactitud train: 0.3536 - Exactitud val: 0.3524
Época 04/50 - Pérdida: 1.5113 - Exactitud train: 0.3119 - Exactitud val: 0.2619
Época 05/50 - Pérdida: 1.4870 - Exactitud train: 0.3464 - Exactitud val: 0.2810
Época 06/50 - Pérdida: 1.4528 - Exactitud train: 0.4143 - Exactitud val: 0.3619
Época 07/50 - Pérdida: 1.4380 - Exactitud train: 0.4119 - Exactitud val: 0.3857
Época 08/50 - Pérdida: 1.4072 - Exactitud train: 0.3560 - Exactitud val: 0.2952
Época 09/50 - Pérdida: 1.3935 - Exactitud train: 0.3548 - Exactitud val: 0.3095
Época 10/50 - Pérdida: 1.3662 - Exactitud train: 0.4036 - Exactitud val: 0.3381
Época 11/50 - Pérdida: 1.3445 - Exactitud train: 0.4083 - Exactitud val: 0.3810
Época 12/50 - Pérdida: 1.3379 - Exactitud train: 0.5226 - Exactitud val: 0.3952
Época 13/50 - Pérdida: 1.3013 - Exactitu

## Celda N°9 – Evaluación final, conteo de errores y matriz de confusión


Interpretación de la matriz de confusión:


In [ ]:
def construir_matriz_confusion(etiquetasReales, etiquetasPredichas, numeroClases):
    """
    Construye una matriz de confusión de tamaño (numeroClases x numeroClases).

    Filas: clases reales
    Columnas: clases predichas

    matriz[i, j] = número de ejemplos de clase real i predichos como clase j.
    """
    matriz = np.zeros((numeroClases, numeroClases), dtype=np.int32)
    for real, pred in zip(etiquetasReales, etiquetasPredichas):
        matriz[real, pred] += 1
    return matriz

def imprimir_matriz_confusion(matrizConfusion, nombresClases, titulo="Matriz de confusión"):
    """
    Imprime la matriz de confusión en formato tabular simple.

    - matrizConfusion: array (numeroClases, numeroClases)
    - nombresClases: lista de nombres de clases
    - titulo: texto a mostrar antes de la tabla
    """
    numeroClases = len(nombresClases)

    print(f"\n{titulo}")
    print("Filas: clase real | Columnas: clase predicha\n")

    # Encabezado
    encabezado = "real\\pred".ljust(12)
    for nombreClase in nombresClases:
        encabezado += nombreClase[:8].ljust(10)
    print(encabezado)

    # Filas
    for i, nombreClaseReal in enumerate(nombresClases):
        fila = nombreClaseReal[:8].ljust(12)
        for j in range(numeroClases):
            fila += str(matrizConfusion[i, j]).ljust(10)
        print(fila)


# Exactitud final
exactitudFinalTrain = calcular_exactitud(modeloCnn, imagenesEntrenamiento, etiquetasEntrenamiento)
exactitudFinalVal = calcular_exactitud(modeloCnn, imagenesValidacion, etiquetasValidacion)

print(f"Exactitud final entrenamiento: {exactitudFinalTrain:.4f}")
print(f"Exactitud final validación: {exactitudFinalVal:.4f}")


# Predicciones en entrenamiento y validación
clasesPredTrain, _ = modeloCnn.predecir_clase_probas(imagenesEntrenamiento)
clasesPredVal, _ = modeloCnn.predecir_clase_probas(imagenesValidacion)

# Conteo de errores
erroresTrain = np.sum(clasesPredTrain != etiquetasEntrenamiento)
erroresVal = np.sum(clasesPredVal != etiquetasValidacion)

print(f"Errores en entrenamiento: {erroresTrain} de {imagenesEntrenamiento.shape[0]}")
print(f"Errores en validación: {erroresVal} de {imagenesValidacion.shape[0]}")


# Matrices de confusión
matrizConfusionTrain = construir_matriz_confusion(
    etiquetasReales=etiquetasEntrenamiento,
    etiquetasPredichas=clasesPredTrain,
    numeroClases=numeroClases
)

matrizConfusionVal = construir_matriz_confusion(
    etiquetasReales=etiquetasValidacion,
    etiquetasPredichas=clasesPredVal,
    numeroClases=numeroClases
)

imprimir_matriz_confusion(matrizConfusionTrain, nombresClases, titulo="Matriz de confusión (entrenamiento)")

imprimir_matriz_confusion(matrizConfusionVal, nombresClases, titulo="Matriz de confusión (validación)")

Exactitud final entrenamiento: 0.7702
Exactitud final validación:    0.4238
Errores en entrenamiento: 193 de 840
Errores en validación:    121 de 210

Matriz de confusión (entrenamiento)
Filas: clase real | Columnas: clase predicha

real\pred   Aves      Caballos  Gatos     Peces     Perros    
Aves        124       6         5         28        9         
Caballos    11        134       8         17        2         
Gatos       1         2         136       19        11        
Peces       4         1         7         143       6         
Perros      13        5         24        14        110       

Matriz de confusión (validación)
Filas: clase real | Columnas: clase predicha

real\pred   Aves      Caballos  Gatos     Peces     Perros    
Aves        10        1         3         17        7         
Caballos    9         17        5         4         3         
Gatos       3         2         15        8         13        
Peces       6         3         6         30        4    

In [ ]:
# Celda X: Exportar modelo entrenado a disco

# Ruta donde se guardará el archivo .npz con los parámetros del modelo
RUTA_MODELO = Path("/content/drive/MyDrive/modelos/cnn_animales_32x32.npz")
RUTA_MODELO.parent.mkdir(parents=True, exist_ok=True)  # crea la carpeta si no existe

def exportar_modelo_npz(modelo, nombresClases, ruta_archivo):
    """
    Guarda en un archivo .npz:
    - Parámetros entrenados de la CNN (filtros y sesgos de la conv, pesos y sesgos de las densas).
    - Información básica del modelo (inputShape, numeroClases).
    - Nombres de las clases.
    - Historial de pérdida y exactitud (train/val) para referencia.

    Para cargarlo en otro notebook:
    >>> datos = np.load(ruta_archivo, allow_pickle=True)
    >>> datos["conv_filtros"], datos["fcOculta_pesos"], ...

    (La reconstrucción del modelo se haría creando un ModeloCNN con la misma
     arquitectura y asignando manualmente estos arrays a cada capa.)
    """
    np.savez(
        ruta_archivo,
        # info del modelo
        inputShape=np.array(modelo.inputShape),
        numeroClases=np.array([modelo.numeroClases], dtype=np.int32),

        # parámetros de la capa convolucional
        conv_filtros=modelo.convLayer.filtros,
        conv_sesgos=modelo.convLayer.sesgos,

        # parámetros de la capa densa oculta
        fcOculta_pesos=modelo.fcOculta.pesos,
        fcOculta_sesgos=modelo.fcOculta.sesgos,

        # parámetros de la capa densa de salida
        fcSalida_pesos=modelo.fcSalida.pesos,
        fcSalida_sesgos=modelo.fcSalida.sesgos,

        # metadatos útiles
        nombresClases=np.array(nombresClases),
        historialPerdida=np.array(historialPerdida),
        historialExactitudTrain=np.array(historialExactitudTrain),
        historialExactitudVal=np.array(historialExactitudVal),
    )
    print(f"Modelo guardado en: {ruta_archivo}")

# Llamamos a la función para exportar el modelo entrenado actual
exportar_modelo_npz(modeloCnn, nombresClases, RUTA_MODELO)

Modelo guardado en: /content/drive/MyDrive/modelos/cnn_animales_32x32.npz


## Celda N°10 – Clasificación con umbral “ninguna de las anteriores” y prueba con imagen nueva.

In [ ]:
def clasificar_con_umbral(modelo, imagenes, nombresClases, umbralNinguna):
    """
    Clasifica imágenes usando la CNN y aplica un umbral de confianza.
    Si la probabilidad máxima < umbralNinguna -> 'ninguna_de_las_anteriores'.
    """
    clasesPredichas, probabilidades = modelo.predecir_clase_probas(imagenes)
    resultados = []

    for i in range(imagenes.shape[0]):
        indiceClase = clasesPredichas[i]
        probMax = float(np.max(probabilidades[i]))

        if probMax < umbralNinguna:
            etiqueta = "ninguna_de_las_anteriores"
        else:
            etiqueta = nombresClases[indiceClase]

        resultados.append((etiqueta, probMax))

    return resultados


def preprocesar_imagen_individual(rutaImagen, alturaImagen, anchuraImagen):
    """
    Preprocesa una imagen como en el dataset:
    - Escala de grises
    - Redimensionar
    - Normalizar [0, 1]
    - Añadir dimensión de canal
    - Añadir dimensión de batch
    """
    imagen = Image.open(rutaImagen).convert("L")
    imagen = imagen.resize((anchuraImagen, alturaImagen))
    arregloImagen = np.array(imagen, dtype=np.float32) / 255.0
    arregloImagen = np.expand_dims(arregloImagen, axis=-1)  # canal
    arregloImagen = np.expand_dims(arregloImagen, axis=0)   # batch
    return arregloImagen


# Cargar modelo guardado y clasificar una imagen

# Ruta del modelo entrenado (.npz) guardado con exportar_modelo_npz
RUTA_MODELO = Path("/content/drive/MyDrive/modelos/cnn_animales_32x32.npz")

# Ruta de ejemplo:
RUTA_IMAGEN_PRUEBA = Path("/content/drive/MyDrive/data/ejemplo/FA1.png")

if not RUTA_MODELO.exists():
    print("No se encontró el archivo de modelo:", RUTA_MODELO)
else:
    # Cargar datos del modelo
    datos = np.load(RUTA_MODELO, allow_pickle=True)

    # Recuperar forma de entrada, número de clases y nombres de clases
    inputShape = tuple(datos["inputShape"])
    numeroClases = int(datos["numeroClases"][0])
    nombresClases = datos["nombresClases"].tolist()

    # Reconstruir la arquitectura del modelo
    modeloCnn = ModeloCNN(inputShape=inputShape, numeroClases=numeroClases)

    # Asignar pesos y sesgos desde el archivo
    modeloCnn.convLayer.filtros      = datos["conv_filtros"].astype(np.float32)
    modeloCnn.convLayer.sesgos       = datos["conv_sesgos"].astype(np.float32)
    modeloCnn.fcOculta.pesos         = datos["fcOculta_pesos"].astype(np.float32)
    modeloCnn.fcOculta.sesgos        = datos["fcOculta_sesgos"].astype(np.float32)
    modeloCnn.fcSalida.pesos         = datos["fcSalida_pesos"].astype(np.float32)
    modeloCnn.fcSalida.sesgos        = datos["fcSalida_sesgos"].astype(np.float32)

    print(f"Modelo cargado correctamente desde: {RUTA_MODELO}")

    # Clasificar la imagen de prueba
    if RUTA_IMAGEN_PRUEBA.exists():
        imagenProcesada = preprocesar_imagen_individual(
            rutaImagen=RUTA_IMAGEN_PRUEBA,
            alturaImagen=ALTURA_IMAGEN,
            anchuraImagen=ANCHURA_IMAGEN
        )

        resultados = clasificar_con_umbral(
            modelo=modeloCnn,
            imagenes=imagenProcesada,
            nombresClases=nombresClases,
            umbralNinguna=UMBRAL_NINGUNA
        )

        etiquetaPredicha, probabilidad = resultados[0]
        print(f"Predicción para la imagen de prueba: {etiquetaPredicha} (p = {probabilidad:.4f})")
    else:
        print(
            "La ruta RUTA_IMAGEN_PRUEBA no existe.\n"
            "Cámbiala por una imagen real para probar la predicción."
        )


Modelo cargado correctamente desde: /content/drive/MyDrive/modelos/cnn_animales_32x32.npz
Predicción para la imagen de prueba: Peces (p = 0.9812)
